# Download Datasets

In [1]:
# 1. Install UCI official data fetching toolkit
!pip install ucimlrepo

# 2. Import necessary libraries
import pandas as pd
from ucimlrepo import fetch_ucirepo

# 3. Fetch dataset (ID 697 from your project proposal)
print("Downloading data from UCI, please wait a few seconds...")
dataset = fetch_ucirepo(id=697)

# 4. Extract features (X) and target labels (y)
X = dataset.data.features
y = dataset.data.targets

# 5. Concatenate them horizontally into a complete data table for easy preview
df = pd.concat([X, y], axis=1)

# 6. Print the number of rows, columns, and display the first 5 rows!
print(f"Data loaded successfully! There are {df.shape[0]} student records and {df.shape[1]} variables.")
df.head()

Data loaded successfully! There are 4424 student records and 37 variables.


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


# Data Preprocessing

In [2]:
# 1. Install libraries for encoding and imbalanced data
!pip install category_encoders imbalanced-learn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from category_encoders import TargetEncoder
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# 2. Check for missing values (this dataset is usually clean, but it's standard practice)
X = X.fillna(X.median(numeric_only=True))

# 3. Convert target labels (Dropout, Enrolled, Graduate) into numerical values (0, 1, 2)
le = LabelEncoder()
y_encoded = le.fit_transform(y.values.ravel()) # ravel() is used to flatten the column vector

# 4. Split training and test sets (this must be done before encoding and SMOTE to prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# 5. Differentiate categorical and numerical features
# (For demonstration purposes, features with less than 10 unique values are considered categorical, others numerical)
categorical_cols = [col for col in X_train.columns if X_train[col].nunique() < 10]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

# 6. Target Encoding - for categorical features
te = TargetEncoder(cols=categorical_cols)
X_train[categorical_cols] = te.fit_transform(X_train[categorical_cols], y_train)
X_test[categorical_cols] = te.transform(X_test[categorical_cols]) # Note: Test set can only be transformed

# 7. Standardization (StandardScaler) - for numerical features
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 8. SMOTE for class imbalance (Important: only apply to training set!)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Preprocessing complete!")
print(f"Training set size before SMOTE: {X_train.shape[0]} rows")
print(f"Training set size after SMOTE: {X_train_smote.shape[0]} rows (classes are balanced)")

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------------------------- ------------- 6.3/9.5 MB 32.2 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 31.3 MB/s  0:00:00

   ---------------------------------------- 0/5 [patsy]
   ---------------------------------------- 0/5 [patsy]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5 [statsmodels]
   -------- ------------------------------- 1/5

# Baseline Models

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# ================= 1. Multinomial Logistic Regression =================
# Added multi_class='multinomial' to fit the multiclass classification task
lr = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train_smote, y_train_smote)
y_pred_lr = lr.predict(X_test)

print("--- Multinomial Logistic Regression ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score (weighted): {f1_score(y_test, y_pred_lr, average='weighted'):.4f}\n")

# ================= 2. Random Forest =================
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_smote, y_train_smote)
y_pred_rf = rf.predict(X_test)

print("--- Random Forest ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score (weighted): {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

In [8]:
import pandas as pd

# Convert processed data to DataFrame and save as CSV (removing extra index)
pd.DataFrame(X_train_smote).to_csv('X_train_processed.csv', index=False)
pd.DataFrame(y_train_smote, columns=['Target']).to_csv('y_train_processed.csv', index=False)

pd.DataFrame(X_test).to_csv('X_test_processed.csv', index=False)
pd.DataFrame(y_test, columns=['Target']).to_csv('y_test_processed.csv', index=False)

print("✅ Handover files generated:")
print("1. X_train_processed.csv")
print("2. y_train_processed.csv")
print("3. X_test_processed.csv")
print("4. y_test_processed.csv")

✅ Handover files generated:
1. X_train_processed.csv
2. y_train_processed.csv
3. X_test_processed.csv
4. y_test_processed.csv
